# Audio Normalization - Pitt数据集 (改进版)

## 问题分析

原始方法（-23 LUFS响度归一化）导致大量削波，原因：
- 原始音频响度过低（可能 -40 LUFS 或更低）
- 提升到 -23 LUFS 时增益过大，峰值超过 1.0
- 强制削波会导致音频失真

## 改进方案

采用 **RMS 能量归一化**：
- 基于音频的均方根（RMS）能量
- 更适合语音信号
- 保留动态范围，不会削波
- 同时设置峰值保护（-1 dB headroom）

## 1. Import Libraries

In [1]:
import time
from pathlib import Path
from tqdm import tqdm

import numpy as np
import soundfile as sf

print(f"NumPy version: {np.__version__}")
print(f"soundfile version: {sf.__version__}")

NumPy version: 2.0.2
soundfile version: 0.13.1


## 2. Configuration

In [2]:
# 输入输出路径
input_dir = Path('data/processed/Pitt-vocals-demucs')
output_dir = Path('data/processed/Pitt-vocals-normalized')

# 归一化参数
TARGET_RMS = 0.1         # 目标RMS能量（0-1之间，推荐0.05-0.15）
PEAK_HEADROOM_DB = -1.0  # 峰值余量（dB），防止削波

# 统计信息
control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

print(f"Input directory: {input_dir}")
print(f"Output directory: {output_dir}")
print(f"Target RMS: {TARGET_RMS:.3f}")
print(f"Peak headroom: {PEAK_HEADROOM_DB} dB")
print(f"\nDataset statistics:")
print(f"  Control samples: {len(control_files)}")
print(f"  Dementia samples: {len(dementia_files)}")
print(f"  Total samples: {len(control_files) + len(dementia_files)}")

Input directory: data/processed/Pitt-vocals-demucs
Output directory: data/processed/Pitt-vocals-normalized
Target RMS: 0.100
Peak headroom: -1.0 dB

Dataset statistics:
  Control samples: 242
  Dementia samples: 309
  Total samples: 551


## 3. RMS Energy Normalization Function

In [3]:
def rms_normalize(audio, target_rms=0.1, peak_headroom_db=-1.0):
    """
    基于RMS能量的归一化（适合语音信号）
    
    Args:
        audio: numpy array, shape (samples,) 单声道音频
        target_rms: float, 目标RMS值（0-1之间）
                    0.05: 较安静
                    0.1:  中等（推荐）
                    0.15: 较响
        peak_headroom_db: float, 峰值余量（负数），防止削波
                         -1.0 dB: 保守（推荐）
                         -3.0 dB: 更保守
    
    Returns:
        normalized_audio: 归一化后的音频
        stats: dict, 统计信息
    """
    # 计算原始RMS
    original_rms = np.sqrt(np.mean(audio**2))
    
    # 如果音频几乎是静音，跳过归一化
    if original_rms < 1e-6:
        return audio, {
            'original_rms': 0.0,
            'original_peak': 0.0,
            'gain_db': 0.0,
            'peak_limited': False,
            'final_rms': 0.0,
            'final_peak': 0.0
        }
    
    # 计算所需增益
    gain = target_rms / original_rms
    
    # 应用增益
    normalized_audio = audio * gain
    
    # 检查峰值，应用峰值限制
    peak = np.abs(normalized_audio).max()
    peak_threshold = 10 ** (peak_headroom_db / 20.0)  # 转换为线性值
    
    peak_limited = False
    if peak > peak_threshold:
        # 应用峰值限制
        peak_gain = peak_threshold / peak
        normalized_audio = normalized_audio * peak_gain
        gain = gain * peak_gain
        peak_limited = True
    
    # 计算最终统计
    final_rms = np.sqrt(np.mean(normalized_audio**2))
    final_peak = np.abs(normalized_audio).max()
    gain_db = 20 * np.log10(gain) if gain > 0 else 0.0
    
    stats = {
        'original_rms': float(original_rms),
        'original_peak': float(np.abs(audio).max()),
        'gain_db': float(gain_db),
        'peak_limited': peak_limited,
        'final_rms': float(final_rms),
        'final_peak': float(final_peak)
    }
    
    return normalized_audio, stats

## 4. Batch Processing Function

In [4]:
def batch_normalize(files, output_subdir, target_rms, peak_headroom_db, group_name):
    """
    批量RMS归一化
    
    Args:
        files: list of Path objects, 待处理的音频文件列表
        output_subdir: Path, 输出子目录
        target_rms: float, 目标RMS值
        peak_headroom_db: float, 峰值余量
        group_name: str, 组名（用于显示）
    
    Returns:
        dict: 统计信息
    """
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    # 统计信息
    summary = {
        'total': len(files),
        'processed': 0,
        'skipped': 0,
        'failed': 0,
        'peak_limited_count': 0,
        'stats_list': [],
        'processing_times': []
    }
    
    for audio_file in tqdm(files, desc=f"Normalize {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            summary['skipped'] += 1
            continue
        
        start_time = time.time()
        
        try:
            # 读取音频
            audio, sr = sf.read(str(audio_file))
            
            # 确保是单声道
            if len(audio.shape) > 1:
                audio = np.mean(audio, axis=1)
            
            # RMS归一化
            normalized_audio, stats = rms_normalize(
                audio, target_rms, peak_headroom_db
            )
            
            # 保存
            sf.write(
                str(output_file),
                normalized_audio,
                sr,
                subtype='PCM_16'  # 16-bit PCM
            )
            
            # 记录统计信息
            summary['processed'] += 1
            summary['stats_list'].append(stats)
            if stats['peak_limited']:
                summary['peak_limited_count'] += 1
            summary['processing_times'].append(time.time() - start_time)
            
        except Exception as e:
            summary['failed'] += 1
            print(f"\n❌ Failed: {audio_file.name}: {e}")
    
    return summary

## 5. Analyze Original Data First (Optional but Recommended)

In [5]:
# 分析原始数据的RMS分布，帮助选择合适的target_rms
def analyze_rms_distribution(files, sample_size=50):
    """
    分析音频文件的RMS分布
    """
    import random
    
    sample_files = random.sample(files, min(sample_size, len(files)))
    rms_values = []
    peak_values = []
    
    for audio_file in tqdm(sample_files, desc="Analyzing"):
        try:
            audio, sr = sf.read(str(audio_file))
            if len(audio.shape) > 1:
                audio = np.mean(audio, axis=1)
            
            rms = np.sqrt(np.mean(audio**2))
            peak = np.abs(audio).max()
            
            rms_values.append(rms)
            peak_values.append(peak)
        except:
            continue
    
    if rms_values:
        print(f"\nRMS Statistics (sample of {len(rms_values)} files):")
        print(f"  Mean: {np.mean(rms_values):.6f}")
        print(f"  Std:  {np.std(rms_values):.6f}")
        print(f"  Min:  {np.min(rms_values):.6f}")
        print(f"  Max:  {np.max(rms_values):.6f}")
        
        print(f"\nPeak Statistics:")
        print(f"  Mean: {np.mean(peak_values):.6f}")
        print(f"  Max:  {np.max(peak_values):.6f}")
        
        # 建议目标RMS
        suggested_rms = np.mean(rms_values) * 3  # 提升3倍
        suggested_rms = min(suggested_rms, 0.15)  # 不超过0.15
        print(f"\n💡 Suggested target_rms: {suggested_rms:.3f}")

print("="*60)
print("Analyzing Original Audio Statistics...")
print("="*60)

print("\n📊 Control Group:")
analyze_rms_distribution(control_files, sample_size=50)

print("\n📊 Dementia Group:")
analyze_rms_distribution(dementia_files, sample_size=50)

Analyzing Original Audio Statistics...

📊 Control Group:


Analyzing: 100%|██████████| 50/50 [00:00<00:00, 95.24it/s] 



RMS Statistics (sample of 50 files):
  Mean: 0.035306
  Std:  0.042595
  Min:  0.000039
  Max:  0.252148

Peak Statistics:
  Mean: 0.383145
  Max:  1.000000

💡 Suggested target_rms: 0.106

📊 Dementia Group:


Analyzing: 100%|██████████| 50/50 [00:00<00:00, 75.56it/s]


RMS Statistics (sample of 50 files):
  Mean: 0.023733
  Std:  0.025581
  Min:  0.000665
  Max:  0.104796

Peak Statistics:
  Mean: 0.336713
  Max:  0.999969

💡 Suggested target_rms: 0.071


## 6. Process All Data

In [6]:
overall_start = time.time()

print("\n" + "="*60)
print("Starting RMS Normalization")
print("="*60)

# Process Control group
print("\n📁 Processing Control group...")
control_stats = batch_normalize(
    control_files,
    output_dir / 'Control',
    TARGET_RMS,
    PEAK_HEADROOM_DB,
    'Control'
)

# Process Dementia group
print("\n📁 Processing Dementia group...")
dementia_stats = batch_normalize(
    dementia_files,
    output_dir / 'Dementia',
    TARGET_RMS,
    PEAK_HEADROOM_DB,
    'Dementia'
)

overall_time = time.time() - overall_start

print("\n" + "="*60)
print("Processing Complete!")
print("="*60)


Starting RMS Normalization

📁 Processing Control group...


Normalize Control: 100%|██████████| 242/242 [00:00<00:00, 127965.40it/s]



📁 Processing Dementia group...


Normalize Dementia: 100%|██████████| 309/309 [00:00<00:00, 172345.74it/s]


Processing Complete!


## 7. Statistics Report

In [7]:
def print_stats(summary, group_name):
    """打印统计信息"""
    print(f"\n{'='*60}")
    print(f"{group_name} Group Statistics")
    print(f"{'='*60}")
    print(f"Total files: {summary['total']}")
    print(f"  ✅ Processed: {summary['processed']}")
    print(f"  ⏭️  Skipped (already exists): {summary['skipped']}")
    print(f"  ❌ Failed: {summary['failed']}")
    print(f"  🔒 Peak limited: {summary['peak_limited_count']} ({summary['peak_limited_count']/max(summary['processed'],1)*100:.1f}%)")
    
    if summary['stats_list']:
        # 提取统计数据
        original_rms = [s['original_rms'] for s in summary['stats_list']]
        final_rms = [s['final_rms'] for s in summary['stats_list']]
        final_peak = [s['final_peak'] for s in summary['stats_list']]
        gain_db = [s['gain_db'] for s in summary['stats_list']]
        
        print(f"\nOriginal RMS Statistics:")
        print(f"  Mean: {np.mean(original_rms):.6f}")
        print(f"  Std:  {np.std(original_rms):.6f}")
        print(f"  Range: [{np.min(original_rms):.6f}, {np.max(original_rms):.6f}]")
        
        print(f"\nFinal RMS Statistics:")
        print(f"  Mean: {np.mean(final_rms):.6f} (target: {TARGET_RMS:.3f})")
        print(f"  Std:  {np.std(final_rms):.6f}")
        print(f"  Range: [{np.min(final_rms):.6f}, {np.max(final_rms):.6f}]")
        
        print(f"\nFinal Peak Statistics:")
        print(f"  Mean: {np.mean(final_peak):.3f}")
        print(f"  Max:  {np.max(final_peak):.3f} (threshold: {10**(PEAK_HEADROOM_DB/20):.3f})")
        
        print(f"\nGain Applied:")
        print(f"  Mean: {np.mean(gain_db):.2f} dB")
        print(f"  Range: [{np.min(gain_db):.2f}, {np.max(gain_db):.2f}] dB")
    
    if summary['processing_times']:
        times = np.array(summary['processing_times'])
        print(f"\nProcessing Time:")
        print(f"  Total: {np.sum(times):.2f}s")
        print(f"  Average per file: {np.mean(times):.3f}s")

# 打印统计信息
print_stats(control_stats, 'Control')
print_stats(dementia_stats, 'Dementia')

# 总体统计
print(f"\n{'='*60}")
print("Overall Summary")
print(f"{'='*60}")
total_processed = control_stats['processed'] + dementia_stats['processed']
total_files = control_stats['total'] + dementia_stats['total']
total_peak_limited = control_stats['peak_limited_count'] + dementia_stats['peak_limited_count']

print(f"Total files processed: {total_processed}/{total_files}")
print(f"Peak limited: {total_peak_limited} ({total_peak_limited/max(total_processed,1)*100:.1f}%)")
print(f"Total time: {overall_time:.2f}s ({overall_time/60:.2f} min)")
print(f"\nNormalization settings:")
print(f"  Target RMS: {TARGET_RMS:.3f}")
print(f"  Peak headroom: {PEAK_HEADROOM_DB} dB")
print(f"\nOutput directory: {output_dir}")

# 质量评估
if total_peak_limited / max(total_processed, 1) > 0.5:
    print(f"\n⚠️  WARNING: >50% of files were peak limited.")
    print(f"   Consider reducing TARGET_RMS or increasing PEAK_HEADROOM_DB.")
elif total_peak_limited == 0:
    print(f"\n✅ EXCELLENT: No clipping detected!")
else:
    print(f"\n✅ GOOD: Minimal clipping ({total_peak_limited/max(total_processed,1)*100:.1f}%)")


Control Group Statistics
Total files: 242
  ✅ Processed: 0
  ⏭️  Skipped (already exists): 242
  ❌ Failed: 0
  🔒 Peak limited: 0 (0.0%)

Dementia Group Statistics
Total files: 309
  ✅ Processed: 0
  ⏭️  Skipped (already exists): 309
  ❌ Failed: 0
  🔒 Peak limited: 0 (0.0%)

Overall Summary
Total files processed: 0/551
Peak limited: 0 (0.0%)
Total time: 0.01s (0.00 min)

Normalization settings:
  Target RMS: 0.100
  Peak headroom: -1.0 dB

Output directory: data/processed/Pitt-vocals-normalized

✅ EXCELLENT: No clipping detected!


## 8. Verification - Check Output Quality

In [8]:
def verify_normalization(audio_path):
    """验证归一化质量"""
    audio, sr = sf.read(str(audio_path))
    
    # 确保单声道
    if len(audio.shape) > 1:
        audio = np.mean(audio, axis=1)
    
    rms = np.sqrt(np.mean(audio**2))
    peak = np.abs(audio).max()
    
    return rms, peak

# 随机抽样检查5个Control和5个Dementia样本
print("\n" + "="*60)
print("Verification - Random Sample Check")
print("="*60)

import random

# 检查Control样本
output_control_files = list((output_dir / 'Control').glob('*.wav'))
if output_control_files:
    print("\n📊 Control Samples (random 5):")
    for audio_file in random.sample(output_control_files, min(5, len(output_control_files))):
        rms, peak = verify_normalization(audio_file)
        print(f"  {audio_file.name:20s} | RMS: {rms:.6f} | Peak: {peak:.3f}")

# 检查Dementia样本
output_dementia_files = list((output_dir / 'Dementia').glob('*.wav'))
if output_dementia_files:
    print("\n📊 Dementia Samples (random 5):")
    for audio_file in random.sample(output_dementia_files, min(5, len(output_dementia_files))):
        rms, peak = verify_normalization(audio_file)
        print(f"  {audio_file.name:20s} | RMS: {rms:.6f} | Peak: {peak:.3f}")

peak_threshold = 10 ** (PEAK_HEADROOM_DB / 20.0)
print(f"\n✅ Expected RMS: ~{TARGET_RMS:.3f} (may be lower if peak limited)")
print(f"✅ Expected Peak: ≤{peak_threshold:.3f}")


Verification - Random Sample Check

📊 Control Samples (random 5):
  128-3.wav            | RMS: 0.067806 | Peak: 0.484
  267-0.wav            | RMS: 0.063837 | Peak: 0.833
  150-1.wav            | RMS: 0.060467 | Peak: 0.885
  002-3.wav            | RMS: 0.064042 | Peak: 0.709
  192-0.wav            | RMS: 0.018486 | Peak: 0.561

📊 Dementia Samples (random 5):
  164-2.wav            | RMS: 0.045922 | Peak: 0.891
  640-0.wav            | RMS: 0.086105 | Peak: 0.891
  338-0.wav            | RMS: 0.090643 | Peak: 0.891
  181-1.wav            | RMS: 0.070590 | Peak: 0.891
  289-2.wav            | RMS: 0.091438 | Peak: 0.891

✅ Expected RMS: ~0.100 (may be lower if peak limited)
✅ Expected Peak: ≤0.891


## 9. Visualization

In [9]:
# 可视化RMS分布对比
import matplotlib.pyplot as plt

if control_stats['stats_list'] or dementia_stats['stats_list']:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Control组 - Original RMS
    if control_stats['stats_list']:
        original_rms_control = [s['original_rms'] for s in control_stats['stats_list']]
        final_rms_control = [s['final_rms'] for s in control_stats['stats_list']]
        
        axes[0, 0].hist(original_rms_control, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        axes[0, 0].set_xlabel('RMS')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].set_title('Control - Original RMS Distribution')
        axes[0, 0].grid(alpha=0.3)
        
        axes[0, 1].hist(final_rms_control, bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
        axes[0, 1].axvline(TARGET_RMS, color='red', linestyle='--', linewidth=2, label=f'Target: {TARGET_RMS:.3f}')
        axes[0, 1].set_xlabel('RMS')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Control - Normalized RMS Distribution')
        axes[0, 1].legend()
        axes[0, 1].grid(alpha=0.3)
    
    # Dementia组
    if dementia_stats['stats_list']:
        original_rms_dementia = [s['original_rms'] for s in dementia_stats['stats_list']]
        final_rms_dementia = [s['final_rms'] for s in dementia_stats['stats_list']]
        
        axes[1, 0].hist(original_rms_dementia, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
        axes[1, 0].set_xlabel('RMS')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Dementia - Original RMS Distribution')
        axes[1, 0].grid(alpha=0.3)
        
        axes[1, 1].hist(final_rms_dementia, bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
        axes[1, 1].axvline(TARGET_RMS, color='red', linestyle='--', linewidth=2, label=f'Target: {TARGET_RMS:.3f}')
        axes[1, 1].set_xlabel('RMS')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Dementia - Normalized RMS Distribution')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('rms_normalization_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Chart saved: rms_normalization_results.png")
else:
    print("\n⚠️  No data to visualize")


⚠️  No data to visualize


## ✅ Done!

RMS归一化完成。归一化后的数据集位于：
- `data/processed/Pitt-vocals-normalized/Control/`
- `data/processed/Pitt-vocals-normalized/Dementia/`

### 为什么RMS归一化更好？

1. **避免削波失真**: 基于能量而非响度，不会强制放大导致削波
2. **保留动态特征**: 保持音频的相对动态范围
3. **适合语音**: RMS更符合语音信号的特点
4. **峰值保护**: 自动限制峰值，防止失真

### 与响度归一化的区别：

| 方法 | 测量方式 | 适用场景 | 削波风险 |
|------|---------|---------|----------|
| **LUFS响度** | 加权响度 | 广播/音乐 | ⚠️ 高（低响度音频） |
| **RMS能量** | 均方根 | 语音/简单信号 | ✅ 低（有峰值保护） |

### 参数调整建议：

如果峰值限制比例仍然较高（>20%），可以：
1. 降低 `TARGET_RMS`（从 0.1 降到 0.08 或 0.05）
2. 增加 `PEAK_HEADROOM_DB`（从 -1.0 改为 -3.0）

### 下一步：
- 使用归一化后的数据进行特征提取和模型训练
- 对比不同归一化参数对模型性能的影响